In [10]:
# Write your imports here
import numpy as np;
import matplotlib.pyplot as plt

# Physical constants
G_GRAVITY = 9.8  # m/s²
DEFAULT_TRUNK_LENGTH = 0.77  # m
DEFAULT_K_FACTOR = 0.27  # trunk moment arm factor



# Safe Lifting Analysis — Biomechanical Load Evaluation
Incorrect lifting often causes low back pain. This project explores the sagittal-plane lumbar load at L5-S1 using the model from *Biomechanical Waist Comfort Model for Lifting* (PDF).

## 1. Goals:
- Derive and validate lumbar (L5-S1) load model equations.
- Implement a reproducible calculation pipeline.
- Compare predicted erector spinae load `F` with NIOSH threshold (3400 N).
- Define risk metrics and analyze sensitivity across posture and object weight.
- Present results with clear bullets and plots.

    
### 1.2. Statement and motivation

Manual lifting remains a leading cause of lower-back injury. Using the *Biomechanical Waist Comfort Model for Lifting* PDF, we treat the sagittal-plane static case and compute L5-S1 loading.

- Objective: quantify how load weight `G`, body load `M`, and trunk angle `α` affect required muscle force `F`.
- Success criteria: correctly implement formula and show `F` vs 3400 N; deliver performance/risk metrics with visual and numerical support.

## 2. Model equations and assumptions

### 2.1. Equilibrium condition
The net moment around the L5-S1 joint must be zero in static balance.

$$M = 0$$

Muscle force multiply its lever arm equals load moments:

where `F` is spinal muscle force, `L` is trunk lever arm. For simplified model we also analyze:

$$F \cdot L = G \cdot L_1 + F_{\mathrm{body}} \cdot L_{2}$$

Expanded form with trunk (F2) and head/armparts (F3) contributions:

Static moment equilibrium at L5-S1:

$$F \cdot L = G \cdot L_1 + F_2 \cdot L_2 + F_3 \cdot L_3$$


### Geometric relationships
Body and object geometry connect trunk inclination to moment arms:

$$\cos\alpha = \frac{H_2 + h - H_1}{L}$$

$$\sin\alpha = \sqrt{1 - \cos^2\alpha}$$

Moment arm relations:

$$L_1 = L_3 = L \sin\alpha,\qquad L_2 = k \, L \sin\alpha$$

###  Empirical L5-S1 force formula
Empirical lumbar load formula from is:

$$F = (3.42 \cdot M + 9.06 \cdot G) \cdot \sin\alpha$$

with

$$M = m_{body} \, g, \qquad G = m_{object} \, g$$

When object load is zero:

$$G = 0 \quad \Rightarrow \quad F = 3.42 \cdot M \cdot \sin\alpha$$

### NIOSH context
- NIOSH recommended limit: 3400 N at L5-S1.
- Simplified RWL form used as quality check: `RWL = W H D V F A C`.


<div style="display: flex; gap: 8px; align-items: center;">
  <img src="images/Biom1.jpg" alt="Biom1" width="200" height="300" >
  <img src="images/Biom2.jpg" alt="Biom2" width="200" height="300" >
  <img src="images/Biom3.jpg" alt="Biom3" width="200" height="300">
  <img src="images/Biom4.jpg" alt="Biom4" width="200" height="300">
</div>

## Symbols list

- $M$ - body weight force at L5-S1 (N), $M = m_{body} * g$
- $G$ - object force (N), $G = m_{object} * g$
- $g$ - gravity force = 9.8 $m / s^2$
- $F$ - erector spinae muscle force needed for equilibrium (N)
- $L$ - trunk lever arm (m), $L1,L2,L3$: moment arm projections (m)
- $H$ - total height (cm), $H1$: L5-S1 height (cm), $H2$: upper body height (cm), $h$: object height (cm)
- $α$ - trunk angle 
- $k$ - proportion factors
- $j$ - body segment mass fraction

## Worked example
Given:
- `m_body=59 kg`, 
- `m_object=10 kg`, 
- `L=0.77 m`, 
- `α=45°`
- `M=59*9.8=578 N`, 
- `G=10*9.8=98 N`, 
- `sin α=0.707`

Compute:
- `F=(3.42*578 + 9.06*98) * 0.707` = 2025 N


In [ ]:
# 2.1 Equilibrium check
def compute_moments_equilibrium(m_body, m_object, alpha_deg, L=DEFAULT_TRUNK_LENGTH, k=DEFAULT_K_FACTOR, g=G_GRAVITY):
    M = m_body * g
    G = m_object * g
    sin_alpha = np.sin(np.deg2rad(alpha_deg))
    L1 = L3 = L * sin_alpha
    L2 = k * L * sin_alpha
    F2 = 0.30 * M
    F3 = 0.0864 * M
    F_required = (G * L1 + F2 * L2 + F3 * L3) / L
    return {
        'M': M,
        'G': G,
        'alpha_deg': alpha_deg,
        'F_required': F_required,
        'L1': L1,
        'L2': L2,
        'L3': L3,
        'F2': F2,
        'F3': F3,
    }



In [12]:

# Example usage
example_2_1 = compute_moments_equilibrium(m_body=59, m_object=10, alpha_deg=45)
print(f"[2.1] equilibrium F estimate = {example_2_1['F_required']:.1f} N")

[2.1] equilibrium F estimate = 137.7 N


In [13]:
# 2.2 Geometry relationships
H1, H2, h = 62, 96, 50
L = 0.77
cos_alpha = (H2 + h - H1) / L
sin_alpha = np.sqrt(max(0.0, 1 - cos_alpha**2))
L1 = L3 = L * sin_alpha
L2 = k * L * sin_alpha
print('cos_alpha =', cos_alpha)
print('sin_alpha =', sin_alpha)
print('L1=L3 =', L1)
print('L2 =', L2)

cos_alpha = 109.0909090909091
sin_alpha = 0.0
L1=L3 = 0.0
L2 = 0.0


In [14]:
# 2.3 Empirical formula
m_body, m_object = 59, 10
g = 9.8
M = m_body * g
G = m_object * g
alpha_deg = 45
sin_alpha = np.sin(np.deg2rad(alpha_deg))
F_paper = (3.42 * M + 9.06 * G) * sin_alpha
print(f"M={M:.1f} N, G={G:.1f} N, F={F_paper:.1f} N")

M=578.2 N, G=98.0 N, F=2026.1 N


backup ideas


#### Why?
assess risk to spine for different loads and postures.
#### What?
derive formula for erector spinae force `F` and compare with NIOSH limit 3400 N.
#### Assumptions: 
static balance, simplified lever mechanics, 2D sagittal plane.
